In [34]:
import sys

# path that contains the `tone_shift` folder
sys.path.append(r"C:\Users\Admin\Desktop\ai")

from tone_shift.features.feature_extraction import extract_features
from tone_shift.synth.synthesis import synthesize_from_params, make_wav
import os
import numpy as np

In [35]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class TimbreDecoder(nn.Module):
    def __init__(
        self,
        in_dim: int,          # number of conditioning features per frame
        gru_hidden: int = 256,
        gru_layers: int = 2,
        mlp_hidden: int = 256,
        harmonics_dim: int = 60,   # K_h
        noise_dim: int = 65        # K_n
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=in_dim,
            hidden_size=gru_hidden,
            num_layers=gru_layers,
            batch_first=True,
            bidirectional=False,
        )

        self.mlp = nn.Sequential(
            nn.Linear(gru_hidden, mlp_hidden),
            nn.LayerNorm(mlp_hidden),
            nn.ReLU(),
            nn.Linear(mlp_hidden, mlp_hidden),
            nn.ReLU(),
        )

        # Separate heads
        self.harmonics_head = nn.Linear(mlp_hidden, harmonics_dim)
        self.noise_head = nn.Linear(mlp_hidden, noise_dim)
        self.gain_head = nn.Linear(mlp_hidden, 1)

    def forward(self, x, h0=None):
        """
        x: [B, T, in_dim]
        returns:
          harmonic_amps: [B, T, K_h] (normalized, ≥0)
          noise_amps:    [B, T, K_n] (≥0)
          gain:          [B, T, 1]   (~0..1)
        """
        gru_out, h_n = self.gru(x, h0)   # [B, T, gru_hidden]

        z = self.mlp(gru_out)            # [B, T, mlp_hidden]

        harmonics = self.harmonics_head(z)
        noise = self.noise_head(z)
        gain = self.gain_head(z)

        # Make them sensible for synthesis
        # harmonics: positive + normalized across harmonics
        harmonics = F.softplus(harmonics)
        harmonics_sum = harmonics.sum(dim=-1, keepdim=True) + 1e-8
        harmonics = harmonics / harmonics_sum

        # noise: just positive
        noise = F.softplus(noise)

        # gain: 0..1
        gain = torch.sigmoid(gain)

        return harmonics, noise, gain, h_n


In [36]:
class MultiScaleSpectrogramLoss(nn.Module):
    def __init__(self,
                 fft_sizes=(2048, 1024, 512),
                 hop_sizes=(512, 256, 128),
                 win_lengths=(2048, 1024, 512),
                 mag_power: float = 1.0):
        super().__init__()
        assert len(fft_sizes) == len(hop_sizes) == len(win_lengths)
        self.fft_sizes = fft_sizes
        self.hop_sizes = hop_sizes
        self.win_lengths = win_lengths
        self.mag_power = mag_power

    def stft_mag(self, x, fft_size, hop_size, win_length):
        # x: [B, T]
        window = torch.hann_window(win_length, device=x.device)
        X = torch.stft(
            x,
            n_fft=fft_size,
            hop_length=hop_size,
            win_length=win_length,
            window=window,
            center=True,
            pad_mode="reflect",
            return_complex=True,
        )
        mag = torch.abs(X) ** self.mag_power
        return mag

    def forward(self, y_hat, y):
        """
        y_hat, y: [B, T]
        """
        loss = 0.0
        for fft_size, hop, win in zip(self.fft_sizes, self.hop_sizes, self.win_lengths):
            Y_hat = self.stft_mag(y_hat, fft_size, hop, win)
            Y = self.stft_mag(y, fft_size, hop, win)
            loss += F.l1_loss(torch.log1p(Y_hat), torch.log1p(Y))
        return loss / len(self.fft_sizes)


In [37]:
import numpy as np
import torch

SR = 16000
HOP_LENGTH = 512
FRAME_RATE = SR / HOP_LENGTH

def prepare_features_for_decoder(wav_path):
    # If your extract_features currently returns 4 values (f0, loudness, onset, y),
    # use this line:
    # f0, loudness, onset, y = extract_features(wav_path)

    # If you already modified extract_features to ALSO return voicing, use:
    # f0, loudness, onset, y, voicing = extract_features(wav_path)

    f0, loudness, onset, y = extract_features(wav_path)  # <- adjust if needed

    f0 = np.asarray(f0, dtype=np.float32)
    loudness = np.asarray(loudness, dtype=np.float32)
    onset = np.asarray(onset, dtype=np.float32)

    # ---- Clean F0 ----
    # NaNs -> 0
    f0 = np.nan_to_num(f0, nan=0.0)
    # clip to a sane range so we never take log2(0) or log2(huge)
    f0_hz = np.clip(f0, 1.0, 3000.0).astype(np.float32)
    # log F0 relative to A4
    f0_log = np.log2(f0_hz / 440.0).astype(np.float32)

    # ---- Clean other features ----
    loudness = np.nan_to_num(loudness, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    onset = np.nan_to_num(onset, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    # ---- Match lengths ----
    T = min(len(f0_log), len(loudness), len(onset))
    f0_hz = f0_hz[:T]
    f0_log = f0_log[:T]
    loudness = loudness[:T]
    onset = onset[:T]

    # ---- Stack features: [f0_log, loudness, onset] ----
    feats = np.stack([f0_log, loudness, onset], axis=-1)  # [T, 3]

    # Final safety: kill any leftover NaN/inf just in case
    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    # ---- Torch tensors ----
    feats_t = torch.from_numpy(feats).unsqueeze(0)         # [1, T, 3]
    f0_t = torch.from_numpy(f0_hz).unsqueeze(0)            # [1, T]
    audio_t = torch.from_numpy(y.astype(np.float32)).unsqueeze(0)  # [1, N_samples]

    frame_rate = FRAME_RATE

    return feats_t, f0_t, audio_t, frame_rate


In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# in_dim should match number of features returned by prepare_features_for_decoder
in_dim = 4      
K_H = 60         
K_N = 65            



decoder = TimbreDecoder(
    in_dim=3,           # 👈 match features: f0_log, loudness, onset
    harmonics_dim=K_H,
    noise_dim=K_N,
).to(device)


ms_loss = MultiScaleSpectrogramLoss().to(device)
optimizer = torch.optim.Adam(decoder.parameters(), lr=1e-3)

PROJECT_ROOT = r"C:\Users\Admin\Desktop\ai\tone_shift"  # adjust if needed

wav_path = os.path.join(
    PROJECT_ROOT,
    "data", "nsynth-test", "audio",
    "string_acoustic_057-068-100.wav"
)


print("Exists:", os.path.exists(wav_path))  


feats_t, f0_t, audio_t, frame_rate = prepare_features_for_decoder(wav_path)
feats_t = feats_t.to(device)
f0_t = f0_t.to(device)
audio_t = audio_t.to(device)
for epoch in range(200):
    decoder.train()
    optimizer.zero_grad()

    harmonics, noise, gain, _ = decoder(feats_t.to(device))

    # TEMP: disable noise so it must use sinusoidal part
    noise = noise * 0.0

    params = {
        "harmonic_amps": harmonics,
        "noise_amps": noise,
        "gain": gain,
        "optional": {}
    }

    y_hat = synthesize_from_params(params, f0_t.to(device), sr=SR, frame_rate=FRAME_RATE)

    N = min(y_hat.shape[1], audio_t.shape[1])
    y_hat = y_hat[:, :N]
    y_ref = audio_t[:, :N].to(device)

    loss = ms_loss(y_hat, y_ref)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(decoder.parameters(), 1.0)
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, loss = {loss.item():.4f}")


Exists: True
Epoch 10, loss = 0.2006
Epoch 10, loss = 0.2006
Epoch 20, loss = 0.1821
Epoch 20, loss = 0.1821
Epoch 30, loss = 0.1527
Epoch 30, loss = 0.1527
Epoch 40, loss = 0.1410
Epoch 40, loss = 0.1410
Epoch 50, loss = 0.1361
Epoch 50, loss = 0.1361
Epoch 60, loss = 0.1333
Epoch 60, loss = 0.1333
Epoch 70, loss = 0.1307
Epoch 70, loss = 0.1307
Epoch 80, loss = 0.1328
Epoch 80, loss = 0.1328
Epoch 90, loss = 0.1263
Epoch 90, loss = 0.1263
Epoch 100, loss = 0.1185
Epoch 100, loss = 0.1185
Epoch 110, loss = 0.1161
Epoch 110, loss = 0.1161
Epoch 120, loss = 0.1140
Epoch 120, loss = 0.1140
Epoch 130, loss = 0.1134
Epoch 130, loss = 0.1134
Epoch 140, loss = 0.1130
Epoch 140, loss = 0.1130
Epoch 150, loss = 0.1127
Epoch 150, loss = 0.1127
Epoch 160, loss = 0.1125
Epoch 160, loss = 0.1125
Epoch 170, loss = 0.1125
Epoch 170, loss = 0.1125
Epoch 180, loss = 0.1123
Epoch 180, loss = 0.1123
Epoch 190, loss = 0.1122
Epoch 190, loss = 0.1122
Epoch 200, loss = 0.1141
Epoch 200, loss = 0.1141


In [39]:
with torch.no_grad():
    harmonics, noise, gain, _ = decoder(feats_t.to(device))
    params = {
        "harmonic_amps": harmonics,
        "noise_amps": noise,
        "gain": gain,
        "optional": {}
    }
    y_hat = synthesize_from_params(params, f0_t.to(device), sr=SR, frame_rate=frame_rate)

print("y_hat shape:", y_hat.shape)
print("min/max:", y_hat.min().item(), y_hat.max().item())
print("has NaN:", torch.isnan(y_hat).any().item())


y_hat shape: torch.Size([1, 64512])
min/max: -1.0 1.0
has NaN: False


In [43]:
with torch.no_grad():
    harmonics, noise, gain, _ = decoder(feats_t.to(device))
    params = {
        "harmonic_amps": harmonics,
        "noise_amps": noise,
        "gain": gain,
        "optional": {}
    }
    y_hat = synthesize_from_params(params, f0_t.to(device), sr=SR, frame_rate=frame_rate)

make_wav(y_hat, SR)  # ✅ creates test_synth.wav


✅ Audio saved to test_synth.wav


In [ ]:
string_acoustic_057-068-100.wav

In [ ]:
# import torch
# import numpy as np
# from tone_shift.synth.synthesis import synthesize_from_params, make_wav

# SR = 16000
# FRAME_RATE = SR / 512  # if your hop_length is 512
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# def test_synth_constant_f0():
#     B = 1
#     T = 126   # same as your other runs
#     K_H = 60
#     K_N = 65

#     f0_hz = torch.full((B, T), 220.0, device=device)  # A3
#     harmonic_amps = torch.zeros(B, T, K_H, device=device)
#     harmonic_amps[:, :, 0] = 1.0     # all energy in 1st harmonic
#     noise_amps = torch.zeros(B, T, K_N, device=device)
#     gain = torch.ones(B, T, 1, device=device) * 0.3

#     params = {
#         "harmonic_amps": harmonic_amps,
#         "noise_amps": noise_amps,
#         "gain": gain,
#         "optional": {}
#     }

#     y, y_harm, y_noise = synthesize_from_params(
#         params,
#         f0_hz,
#         sr=SR,
#         frame_rate=FRAME_RATE,
#         noise_scale=0.0,          # IMPORTANT: no noise
#         return_parts=True,
#     )
#     print("y min/max:", y.min().item(), y.max().item())
#     make_wav(y, SR)

# test_synth_constant_f0()


y min/max: -0.30000001192092896 0.30000001192092896
✅ Audio saved to test_synth.wav


In [42]:
# import numpy as np
# from tone_shift.features.feature_extraction import extract_features

# wav_path = r"C:\Users\Admin\Desktop\ai\tone_shift\data\nsynth-test\audio\bass_electronic_018-022-100.wav"

# f0, loudness, onset, y = extract_features(wav_path)
# f0 = np.asarray(f0, dtype=np.float32)
# f0 = np.nan_to_num(f0, nan=0.0)

# print("F0 shape:", f0.shape)
# print("min nonzero F0:", np.min(f0[f0 > 0]) if np.any(f0 > 0) else None)
# print("max F0:", np.max(f0))
# print("% voiced frames:", 100 * np.mean(f0 > 0), "%")
